# Compute a Gene Covariance Matrix from Atlas Streams

This tutorial shows how to compute a gene-by-gene covariance matrix from Atlas
expression minibatches in a single pass. It demonstrates how custom statistical
methods can process all selected cells without materializing the complete
cell-by-gene matrix in memory.

A streamed covariance matrix can support exploratory feature-correlation
analysis and small-scale dimensionality-reduction prototypes. For routine PCA,
use the built-in `sap.tl.pca()` workflow.

This tutorial builds on the single-pass streaming pattern introduced in
{doc}`stream-mean-and-variance`. Here, the same idea is extended from
per-gene summaries to a gene-by-gene covariance matrix, where memory planning
and accumulator updates become more important.

By the end of this tutorial, you will be able to:

- traverse an Atlas exactly once using dense minibatches;
- merge batch-level means and covariance accumulators;
- estimate a sample covariance matrix across all selected cells;
- plan memory use for gene-by-gene outputs;
- validate and save the resulting matrix.

## Before You Begin

This tutorial assumes that:

- quality control and preprocessing have been completed;
- an existing Atlas is open;
- the selected expression field is appropriate for covariance analysis;
- the selected number of genes is small enough for a dense
  gene-by-gene matrix.

The examples below assume an existing Atlas object:



In [ ]:
import numpy as np
import scatlaspy as sap

atlas = sap.Atlas(
    "./tmp/tutorials/basic_pbmc3k/pbmc3k_basic_copy.sasql",
    db_memory_limit="8GB",
)



## 1. Define the Analysis View

Build a read index that defines the cells, genes, and expression representation
included in the calculation:



In [ ]:
atlas.build_read_index(
    cell_condition="filter_cells",
    gene_condition="filter_genes",
    use_hvg=True,
    use_data="data_scale",
)



In this example, the covariance matrix is calculated from:

- cells selected by `filter_cells`;
- genes selected by `filter_genes`;
- genes marked as highly variable;
- scaled expression values stored in `data_scale`.

```{important}
The rows and columns of the resulting covariance matrix follow the feature order
defined by the current read index. Save that gene order together with the
matrix. A covariance matrix cannot be interpreted correctly from its shape
alone.
```

## 2. Plan Memory and Computation

For \(p\) selected genes, the covariance matrix contains \(p^2\) values.

The following table shows the approximate size of one dense `float64`
gene-by-gene array:

| Genes | Matrix shape | Size of one `float64` array |
|---:|---:|---:|
| 2,000 | 2,000 × 2,000 | 30.5 MiB |
| 5,000 | 5,000 × 5,000 | 190.7 MiB |
| 10,000 | 10,000 × 10,000 | 762.9 MiB |

The peak memory requirement is larger than the size of the final matrix. During
the calculation, memory may also be required for:

- the global covariance accumulator;
- the covariance accumulator for the current minibatch;
- the dense expression minibatch;
- a centered copy of the minibatch;
- temporary arrays used during matrix operations.

At least two gene-by-gene arrays may coexist during the batch merge. An
eigendecomposition may require another gene-by-gene eigenvector matrix.

```{warning}
Streaming removes the need to store the complete cell-by-gene matrix, but it
does not remove the quadratic memory or computational cost along the gene axis.

For \(n\) cells and \(p\) genes, forming a dense covariance matrix requires
approximately \(O(np^2)\) computation and \(O(p^2)\) memory.
```

Use highly variable genes or another focused feature set unless a much larger
matrix is explicitly required.

## 3. Understand the Batch-wise Merge

For each minibatch, calculate:

- the number of cells in the batch;
- the batch mean;
- the unnormalized within-batch covariance accumulator.

If two groups contain \(n_a\) and \(n_b\) observations, with means
\(\mu_a\) and \(\mu_b\), their accumulators can be merged using:

```{math}
M_{2,a \cup b}
=
M_{2,a}
+
M_{2,b}
+
\frac{n_a n_b}{n_a+n_b}
(\mu_b-\mu_a)(\mu_b-\mu_a)^\mathsf{T}.
```

The merged mean is:

```{math}
\mu_{a \cup b}
=
\mu_a
+
\frac{n_b}{n_a+n_b}
(\mu_b-\mu_a).
```

This merge is numerically more stable than accumulating raw sums of
\(x^\mathsf{T}x\) and subtracting the squared mean at the end.

## 4. Stream and Merge the Minibatches

Initialize the global state:



In [ ]:
n_total = 0
mean = None
m2 = None



Traverse the selected cells once:



In [ ]:
for batch_id, X_batch in enumerate(
    atlas.get_minibatch_dense(
        pass_mode="single-pass",
        batch_size=2048,
    ),
    start=1,
):
    X_batch = np.asarray(X_batch, dtype=np.float64)

    if X_batch.ndim != 2:
        raise ValueError("Each minibatch must be a two-dimensional matrix.")

    n_batch, n_features = X_batch.shape

    if n_batch == 0:
        continue

    if mean is not None and n_features != mean.shape[0]:
        raise ValueError(
            "The feature dimension changed between minibatches."
        )

    batch_mean = X_batch.mean(axis=0)
    centered = X_batch - batch_mean
    batch_m2 = centered.T @ centered

    if n_total == 0:
        n_total = n_batch
        mean = batch_mean.copy()
        m2 = batch_m2
        continue

    delta = batch_mean - mean
    new_total = n_total + n_batch

    # Add the within-batch covariance contribution.
    m2 += batch_m2

    # Reuse batch_m2 as workspace for the between-group correction.
    np.multiply.outer(delta, delta, out=batch_m2)
    batch_m2 *= n_total * n_batch / new_total
    m2 += batch_m2

    mean += delta * n_batch / new_total
    n_total = new_total

    if batch_id == 1 or batch_id % 100 == 0:
        print(f"Processed {n_total:,} cells")



This loop retains:

- one dense expression minibatch;
- its centered representation;
- the global mean vector;
- the gene-by-gene covariance accumulator.

It does not retain expression values from earlier minibatches.

```{note}
The calculation uses `float64` accumulation for numerical stability. Even when
the stored expression values use `float32`, covariance accumulation may benefit
from the additional precision.
```

## 5. Finalize the Covariance Matrix

The accumulator `m2` contains the summed cross-products around the global mean.

Calculate the sample covariance matrix using \(n-1\) in the denominator:



In [ ]:
if n_total < 2:
    raise ValueError(
        "At least two cells are required to calculate covariance."
    )

m2 /= n_total - 1
covariance = m2

print(f"Processed {n_total:,} cells")
print(f"Mean vector shape: {mean.shape}")
print(f"Covariance matrix shape: {covariance.shape}")



This operation reuses the accumulator array rather than allocating another
complete gene-by-gene matrix.

For a population covariance estimate, divide by `n_total` instead. Most
statistical workflows use the sample covariance definition shown above.

## 6. Validate the Result

Check that the output has the expected dimensions and contains finite values:



In [ ]:
if covariance.ndim != 2:
    raise ValueError("The covariance output is not a matrix.")

if covariance.shape[0] != covariance.shape[1]:
    raise ValueError("The covariance matrix is not square.")

if covariance.shape[0] != mean.shape[0]:
    raise ValueError(
        "The mean vector and covariance matrix use different feature counts."
    )

if not np.isfinite(mean).all():
    raise ValueError("The streamed mean contains non-finite values.")

if not np.isfinite(covariance).all():
    raise ValueError("The covariance matrix contains non-finite values.")



Check numerical symmetry:



In [ ]:
max_asymmetry = np.max(np.abs(covariance - covariance.T))

print(f"Maximum asymmetry: {max_asymmetry:.3e}")

if not np.allclose(
    covariance,
    covariance.T,
    rtol=1e-7,
    atol=1e-8,
):
    raise ValueError("The covariance matrix is not numerically symmetric.")



Inspect the variance range:



In [ ]:
variances = np.diag(covariance)

print(f"Minimum variance: {variances.min():.6f}")
print(f"Maximum variance: {variances.max():.6f}")



Small negative eigenvalues or variances can occasionally arise from
floating-point roundoff, but substantial negative values usually indicate a
calculation or data problem.

```{note}
When `use_data="data_scale"`, the covariance diagonal may be near 1 if scaling
was calculated on exactly the same cells and genes, without clipping, and with
a compatible variance convention.

The diagonal may differ from 1 when:

- scaling used a different cell population;
- scaled values were clipped;
- the current read index selects a subset of the scaled cells;
- the scaling and covariance calculations use different degrees of freedom;
- some genes have zero or very small variance.
```

## 7. Compare with an In-memory Calculation

For development and testing, validate the streaming implementation on a small
dataset that fits in memory.

Given a small matrix `X_small`, NumPy calculates the corresponding sample
covariance with:



In [ ]:
reference_covariance = np.cov(
    X_small,
    rowvar=False,
    ddof=1,
)



Compare the streamed and in-memory results only when both calculations use:

- exactly the same cells;
- exactly the same genes;
- exactly the same gene order;
- the same expression representation;
- the same covariance denominator.



In [ ]:
np.testing.assert_allclose(
    covariance,
    reference_covariance,
    rtol=1e-6,
    atol=1e-8,
)



A small reference calculation is one of the most effective ways to test a
custom streaming statistic before applying it to a large Atlas.

## 8. Use the Covariance Matrix

For a moderate number of genes, calculate eigenvalues and eigenvectors with
NumPy:



In [ ]:
eigvals, eigvecs = np.linalg.eigh(covariance)

order = np.argsort(eigvals)[::-1]
eigvals = eigvals[order]
eigvecs = eigvecs[:, order]



Because covariance matrices are symmetric, `np.linalg.eigh()` is more
appropriate than the general `np.linalg.eig()` function.

Inspect the leading eigenvalues:



In [ ]:
print(eigvals[:10])



```{warning}
A complete eigendecomposition has approximately \(O(p^3)\) computational cost
and returns a dense \(p \times p\) eigenvector matrix.

For large feature sets, use truncated, randomized, incremental, or
matrix-free methods rather than calculating every eigenvector of the full
covariance matrix.
```

For routine atlas-scale dimensionality reduction, use the built-in PCA
workflow:



In [ ]:
sap.tl.pca(
    atlas,
    n_components=50,
)



## 9. Save the Result

Save the mean vector and covariance matrix:



In [ ]:
from pathlib import Path

output_path = Path("./results/streamed_covariance.npz")
output_path.parent.mkdir(parents=True, exist_ok=True)

np.savez(
    output_path,
    n_cells=n_total,
    mean=mean,
    covariance=covariance,
)

print(f"Saved covariance result to {output_path}")



Also save or record:

- the Atlas used for the calculation;
- the selected cells;
- the selected genes and their exact order;
- the expression field;
- the filtering and HVG settings;
- the covariance denominator;
- the batch size and software version.

Without the associated gene order, the rows and columns of the covariance
matrix cannot be mapped reliably back to biological features.

## Limitations

This approach is useful when the number of cells is large but the selected
feature set is moderate.

It does not eliminate:

- quadratic storage in the number of genes;
- quadratic computation per cell;
- the cost of dense expression minibatches;
- the cubic cost of a complete eigendecomposition.

For very large feature sets, consider:

- reducing the feature set;
- computing selected gene-gene relationships only;
- randomized or incremental dimensionality reduction;
- low-rank sketches;
- sparse or block-structured approximations;
- matrix-free optimization methods.

## Next Steps

See {doc}`implement-minibatch-kmeans` for an iterative method that uses
randomized, multi-pass minibatches rather than a deterministic single-pass
summary.

See {doc}`stream-mean-and-variance` for a linear-memory streaming statistic that
does not require a gene-by-gene output matrix.